<a href="https://colab.research.google.com/github/luisavgonzalez27/ads-509-Final-Project/blob/main/01_MAUDE_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAUDE Medical Device Adverse Event Data Collection

## ADS-509 Final Project

This notebook collects medical device adverse event reports from the FDA MAUDE database using the openFDA API.

The project focuses on diabetes related medical devices, specifically continuous glucose monitoring (CGM) systems. The collected event narratives will be used to evaluate whether natural language processing and large language models can classify adverse events based on their textual descriptions.

### Initial Research Question

Can an LLM accurately classify the type of adverse event associated with a continuous glucose monitoring device based on the MAUDE event narrative?

### Data Source

FDA Manufacturer and User Facility Device Experience (MAUDE) database via the openFDA Device Adverse Event API.

In [2]:
import requests
import pandas as pd
import json

In [3]:
#setup the API request
# openFDA Medical Device Adverse Event API
BASE_URL = "https://api.fda.gov/device/event.json"

# Search for CGM reports using FDA product code QBJ
params = {
    "search": 'device.device_report_product_code:"QBJ"',
    "limit": 1
}

response = requests.get(BASE_URL, params=params)

print(response.status_code)

200


In [4]:
#inspecting the raw report
data = response.json()

print(json.dumps(data["results"][0], indent=2))

{
  "manufacturer_contact_zip_ext": "",
  "manufacturer_g1_address_2": "",
  "event_location": "I",
  "report_to_fda": "N",
  "manufacturer_contact_t_name": "",
  "manufacturer_contact_state": "",
  "manufacturer_link_flag": "Y",
  "manufacturer_contact_address_2": "",
  "manufacturer_g1_city": "",
  "manufacturer_contact_address_1": "",
  "manufacturer_contact_pcity": "",
  "event_type": "Malfunction",
  "report_number": "3004753838-2020-038805",
  "type_of_report": [
    "Initial submission"
  ],
  "product_problem_flag": "Y",
  "date_received": "20200427",
  "manufacturer_address_2": "",
  "pma_pmn_number": "DEN170088",
  "date_of_event": "20200418",
  "reprocessed_and_reused_flag": "N",
  "manufacturer_address_1": "",
  "exemption_number": "",
  "manufacturer_contact_zip_code": "",
  "reporter_occupation_code": "OTHER",
  "manufacturer_contact_plocal": "",
  "noe_summarized": "1",
  "manufacturer_contact_l_name": "",
  "source_type": [
    "DISTRIBUTOR",
    "FOREIGN"
  ],
  "distr

In [5]:
#extracting useful information:
# Get the first report
report = data["results"][0]

# Find the event/problem description
for text_entry in report["mdr_text"]:
    if text_entry["text_type_code"] == "Description of Event or Problem":
        narrative = text_entry["text"]

print(narrative)

IT WAS REPORTED THAT EARLY SENSOR EXPIRATION OCCURRED. NO PRODUCT OR DATA WAS PROVIDED FOR EVALUATION. CONFIRMATION OF THE ALLEGATION AND A PROBABLE CAUSE COULD NOT BE DETERMINED. NO INJURY OR MEDICAL INTERVENTION WAS REPORTED.


In [7]:
# Extract important fields from the report

report_id = report.get("report_number")
event_type = report.get("event_type")
date_received = report.get("date_received")

device = report.get("device", [{}])[0]

brand_name = device.get("brand_name")
generic_name = device.get("generic_name")
manufacturer = device.get("manufacturer_d_name")
product_code = device.get("device_report_product_code")

product_problems = report.get("product_problems", [])

patient = report.get("patient", [{}])[0]
patient_problems = patient.get("patient_problems", [])

clean_report = {
    "report_id": report_id,
    "date_received": date_received,
    "brand_name": brand_name,
    "generic_name": generic_name,
    "manufacturer": manufacturer,
    "product_code": product_code,
    "narrative": narrative,
    "event_type": event_type,
    "product_problems": product_problems,
    "patient_problems": patient_problems
}

df = pd.DataFrame([clean_report])

df

,report_id,date_received,brand_name,generic_name,manufacturer,product_code,narrative,event_type,product_problems,patient_problems
0,3004753838-2020-038805,20200427,DEXCOM G6 CONTINUOUS GLUCOSE MONITORING SYSTEM,CONTINUOUS GLUCOSE MONITOR,"DEXCOM, INC.",QBJ,IT WAS REPORTED THAT EARLY SENSOR EXPIRATION O...,Malfunction,"[Premature End-of-Life Indicator, Premature En...","[No Consequences Or Impact To Patient, No Cons..."
